#AI Knowledge Graph Builder for Enterprise Intelligence

#ISRO Mission Navigator

##MODULE 4 — RAG + SEMANTIC SEARCH

1.   RAG over original ISRO dataset text
2.   RAG over original ISRO dataset text
1.   Using Hugging Face (Free model)
2.   FAISS
1.   LangChain


###STEP 1 — Install Required Libraries

In [3]:
# Install updated modular LangChain ecosystem
!pip install langchain==0.1.20 \
langchain-community==0.0.38 \
langchain-core==0.1.52 \
langchain-text-splitters==0.0.1 \
langchain-huggingface==0.0.3 \
faiss-cpu \
sentence-transformers \
transformers \
accelerate

###STEP 2 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import faiss

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

from transformers import pipeline

###STEP 3 — Load Data Sources

1. Raw ISRO Dataset Text

2. Graph Triples CSV

In [4]:
#Load Graph Triples
triples_df = pd.read_csv("/content/exported_graph.csv")
triples_df.head()

,source,relationship,target
0,GSAT-15,USED_FOR,"Communication, Navigation"
1,GSAT-15,LAUNCH_ON,11-11-2015
2,GSAT-15,ORBIT_TYPE,GEO
3,GSAT-15,LAUNCHED_BY,Ariane-5 VA-227
4,GSAT-15,used_for,"Communication, Navigation"


In [5]:
print(triples_df.columns)

Index(['source', 'relationship', 'target'], dtype='object')


In [6]:
#Convert triples to readable text:
triples_df["text"] = (
    triples_df["source"].astype(str) + " " +
    triples_df["relationship"].astype(str) + " " +
    triples_df["target"].astype(str)
)

texts = triples_df["text"].tolist()

###STEP 4 — Split into Chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.create_documents(texts)

###STEP 5 — Load FREE Embedding Model (HuggingFace)

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

###STEP 6 — Create FAISS Vector Stores

In [9]:
vector_store = FAISS.from_documents(docs, embedding_model)

###STEP 7 — Load Free LLM (HuggingFace)

In [10]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline

# Correct & stable configuration
hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=False   # <-- IMPORTANT (instead of temperature=0)
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM

###STEP 8 — Create RAG Pipelines

In [11]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever()
)

###STEP 9 — Test Semantic Queries

In [12]:
response = qa_chain.run(
    "Which rocket launched Chandrayaan-3?"
)

print(response)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Chandrayaan2 LAUNCHED_BY GSLV-Mk III - M1 / Chandrayaan-2 Mission

Chandrayaan2 launched_by GSLV-Mk III - M1 / Chandrayaan-2 Mission

Chandrayaan-3 LAUNCHED_BY LVM3 M4 / Chandrayaan-3 Mission

Chandrayaan-3 launched_by LVM3 M4 / Chandrayaan-3 Mission

Question: Which rocket launched Chandrayaan-3?
Helpful Answer:


In [13]:
from langchain_core.documents import Document

# Convert triples into documents
triple_docs = [
    Document(page_content=f"{row['source']} {row['relationship']} {row['target']}")
    for index, row in triples_df.iterrows()
]

# Create FAISS index for graph triples
vector_store_graph = FAISS.from_documents(
    documents=triple_docs,
    embedding=embedding_model
)

print("Graph FAISS index created successfully!")

Graph FAISS index created successfully!


###STEP 10 — Save FAISS Index

In [14]:
vector_store.save_local("faiss_graph_index")
print("FAISS indexes saved successfully!")

FAISS indexes saved successfully!
